# Generators / streaming iterators

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### Generators

> **Problem.** The ingestion job reads a 50 GB JSONL export of support tickets to embed them. `json.load(open(path))` tries to hold all 50 GB in memory; the machine has 16, and the job dies at 30%. Even on a bigger machine, nothing happens for 20 minutes while the file loads before the first ticket is processed.

**Idea.** A function that hands out one item at a time, so a pipeline processes data far larger than memory and starts producing results immediately.

**Use when** ingestion, ETL, logs — anything bigger than memory or where the first result should arrive early.  
**Not when** you need to read the data twice or index into it — build a list or a table.

```mermaid
flowchart LR
    F[(50 GB file)] --> R[read one line] --> K[filter] --> T[transform] --> W[write]
    W -.->|next| R
```

**How it works.**
1. `read_jsonl(path)` contains `yield json.loads(line)`: it hands out one parsed line and pauses until the next is requested.
2. `only_published(rows)` takes that stream and yields only the rows it keeps; `with_word_count(rows)` yields each row with a field added.
3. Chaining them — `with_word_count(only_published(read_jsonl(path)))` — runs nothing. It only builds the pipeline.
4. The `for` loop at the end pulls one row: the pull travels back through the chain, one line is read, filtered, transformed, and handed out.
5. At any moment exactly one row is alive in the pipeline; memory does not grow with the file.

| | what happens | result |
|:--|:--|:--|
| ✗ read() | whole file into memory | 50 GB → crash |
| ✓ generator chain | one row at a time | 33,334 rows processed, ~1 KB memory |

**Production code and its real output**

In [2]:
# Generators — a streaming file pipeline. Each stage pulls one row from the previous one, so a
# 50 GB JSONL file never has to fit in memory. This is how ingestion jobs are written.
import tempfile
from collections.abc import Iterator
from pathlib import Path


def read_jsonl(path: Path) -> Iterator[dict]:
    with path.open() as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def only_published(rows: Iterator[dict]) -> Iterator[dict]:
    for row in rows:
        if row["status"] == "published":
            yield row


def with_word_count(rows: Iterator[dict]) -> Iterator[dict]:
    for row in rows:
        yield {**row, "words": len(row["text"].split())}


with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "docs.jsonl"
    with path.open("w") as handle:
        for index in range(100_000):
            status = "published" if index % 3 == 0 else "draft"
            handle.write(
                json.dumps({"id": index, "status": status, "text": "word " * (index % 20 + 1)})
                + "\n"
            )

    pipeline = with_word_count(only_published(read_jsonl(path)))  # lazy: nothing read yet
    processed = 0
    for _row in pipeline:  # rows flow through one at a time
        processed += 1

print(f"file {path.name}: {processed} published rows streamed through the pipeline")
assert processed == 33_334

file docs.jsonl: 33334 published rows streamed through the pipeline


**What the output shows.** 100,000 rows were written to a temporary file and 33,334 (every third, the published ones) flowed through the three-stage pipeline one at a time.

**In practice**
- **checkpoint** — record the line offset every N rows so a crash at row 40 million resumes there instead of restarting a 3-hour job.
- **batch the I/O** — yield rows one at a time but write them downstream in batches of 500–1,000; one database insert per row is the usual bottleneck.
- **single pass** — a generator is consumed once; if a later stage needs the data again, that is a sign to materialise it (a table, a file).
- **errors surface late** — a bad line 30 GB in fails when it is reached, not when the pipeline is built — wrap stages so one bad row is logged and skipped.
- **the list() trap** — someone adds `rows = list(rows)` "to see how many there are" and the 50 GB job is back.

**Alternatives** — chunked reading with pandas/polars (`chunksize`) when you need table operations · a streaming framework (Spark, Beam) at cluster scale

**Terms** — *yield*: hand out one value, pause until the next is asked for · *lazy*: nothing runs until something asks · *single pass*: used up after one loop


### streaming iterators

> **Problem.** A support chatbot answers in 3–4 seconds. Users see a blank box, assume it is broken, and retype the question — doubling the load. The answers were fine; the wait was the problem.

**Idea.** Consume the model's tokens as they are produced; the first words appear in half a second and the reply reads itself out.

**Use when** a person is waiting: chat, voice, live UI.  
**Not when** batch jobs — no one is watching, streaming only adds code and complexity.

```
without   [ ········ wait 3 s ········ ] ──▶ full answer
with      The  ocean  covers  most  of … ──▶ first word 0.5 s, last 3 s
```

**How it works.**
1. `client.chat.completions.stream(...)` opens a connection that stays open while the model generates.
2. The provider sends a small chunk each time a token (roughly a word) is ready, over server-sent events.
3. The generator `stream_answer` loops over those chunks and `yield`s just the text of each one.
4. The caller — here a `for` loop printing, in a service an SSE endpoint — receives text within ~0.5 s and keeps receiving until the last chunk.
5. The `with` block closes the connection when the loop ends, or when the caller stops early.

| | what happens | result |
|:--|:--|:--|
| ✗ wait | 3 s of nothing, then text | feels broken |
| ✓ stream | first token 0.61 s, done 1.30 s | feels instant, same cost |

**Production code and its real output**

In [3]:
# Streaming iterators — forward tokens as they arrive. The generator yields text deltas; the
# caller (a chat UI, an SSE endpoint) decides what to do with them.
import time
from collections.abc import Iterator


def stream_answer(prompt: str) -> Iterator[str]:
    with client.chat.completions.stream(
        model=settings.openai_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=80,
    ) as stream:
        for event in stream:
            if event.type == "content.delta":
                yield event.delta


started = time.perf_counter()
first_token_at = None
text = ""
for delta in stream_answer("Write two short sentences about the ocean."):
    first_token_at = first_token_at or time.perf_counter() - started
    text += delta
    print(delta, end="", flush=True)
print(f"\nfirst token {first_token_at:.2f}s, done {time.perf_counter() - started:.2f}s")
assert len(text) > 20

The

 ocean

 is

 a

 vast

...

 planet

's

 climate

.


first token 0.69s, done 1.07s


**What the output shows.** The first token arrived well before the last one. With a normal call the user would have seen nothing until the final timestamp.

**In practice**
- **client disconnect** — if the user closes the tab, stop reading and close the stream; the provider keeps generating and billing until you do.
- **every hop** — one proxy or gateway that buffers responses turns streaming back into a 3-second wait. Check nginx, load balancers and the browser fetch.
- **measure both** — time-to-first-token is the UX metric; total time is the cost metric. Track them separately.
- **tool calls** — when the model streams a function call, the arguments arrive in fragments; collect them until the call is complete before parsing JSON.
- **usage at the end** — token counts arrive in the final chunk (or not at all unless requested); cost tracking must read that last event.

**Alternatives** — a non-streaming call with a progress indicator (simple, still 3 s of waiting) · server-side buffering that emits progress events for long jobs

**Terms** — *token*: a piece of a word; models write one at a time · *TTFT*: time to first token — what users feel · *SSE*: server-sent events, the web technique that pushes tokens to a browser
